# CVRIE — modèle amélioré

Prototype expérimental sans diagnostic. Le test est une cohorte déjà observée. Les gains restent exploratoires.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "app.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.common.paths import DATA
from src.supervised.high_resolution import prepare_high_resolution
manifest = prepare_high_resolution()
manifest.groupby(["split", "state"]).size().unstack()


## Validation croisée et seuil
Six configurations comparées sur Train. Seuil choisi sur Validation. Les groupes de patients de référence sont conservés.


In [ ]:
from src.supervised.improve import main
report = main()
import pandas as pd
pd.DataFrame(report["cross_validation"]).T[["auc_mean", "auc_std"]]


In [ ]:
pd.DataFrame({"Avant": report["baseline"]["test"], "Maintenant": report["test"]}).loc[["roc_auc", "accuracy", "precision", "sensitivity", "specificity"]]


## Inférence
Score de classification, pas probabilité personnelle de cancer.


In [ ]:
from src.supervised.inference import load_model, predict
example = manifest[manifest.split == "Test"].iloc[0]
result = predict((DATA / example.path).read_bytes(), load_model())
result


## Extension publique LIDC-IDRI
42 nouveaux patients, 288 images au total. Voir `docs/EXTENDED_DATA.md`.
Deux recherches supplémentaires (40 configurations, 600 ajustements CV) n’ont pas confirmé de gain régulier ; le modèle JSRT précédent reste actif.
Reproduction depuis la racine : `python -m src.supervised.extend_dataset`, puis `python -m src.supervised.train_extended` et `python -m src.supervised.train_extended_512`.
Les tests ont désormais été consultés : leurs résultats restent exploratoires.